In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import pandas as pd 

In [3]:
df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth =df_ground_truth.to_dict(orient = "records")

In [4]:
print(ground_truth[:2])

[{'question': 'Is it okay to join the course late if I just found it now?', 'document': '74eb249bbf'}, {'question': 'Can I still take this course even if I missed the start date?', 'document': '74eb249bbf'}]


In [5]:
from ingest import load_faq_data,build_index

In [6]:
faq_data = load_faq_data()
documents = []
for document in faq_data: 
    if document["course"] == "llm-zoomcamp":
        documents.append(document)

index = build_index(documents)


In [7]:
index


In [8]:
doc_idx = {}
for doc in documents:
    doc_idx[doc["id"]]= doc

In [9]:
from dotenv import load_dotenv
from google import genai

In [10]:
load_dotenv()

True

In [11]:
client = genai.Client()

In [12]:
from evaluation_utils import RAGWithUsage
assistant = RAGWithUsage(
    index = index, 
    llm_client=client
)

In [13]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [14]:
assistant.total_cost()

0.000993

In [15]:
360

360

In [16]:
len(ground_truth)* assistant.total_cost()

0.392235

In [17]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [18]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [20]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

    return result

In [2]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [3]:
df_answers.head(5)

,question,answer_llm,answer_orig,document
0,Is it okay to join the course late if I just f...,"Yes, you can still join the course late. If yo...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,Can I still take this course even if I missed ...,"Yes, you can still join if you missed the star...","Yes, but if you want to receive a certificate,...",74eb249bbf
2,If I join after the course has already started...,"Yes, as long as you join while the course is s...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,Do I need to submit my project before submissi...,"Yes — to get the certificate, you need to subm...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,I’m a bit late to the course—what do I need to...,"To still earn the certificate, you need to:\n\...","Yes, but if you want to receive a certificate,...",74eb249bbf


In [4]:
#A->Q->A' evaluation

In [5]:
# We'll compare the RAG answer with the original answer from the FAQ. This checks if the RAG pipeline is producing answers that match the ground truth.

# First, define the output format:

In [6]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [7]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [8]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [10]:
from dotenv import load_dotenv
from google import genai
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
client = genai.Client()

In [11]:
rec = answers[0]

In [12]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [13]:
eval_result, usage = llm_structured_retry(
    client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning="The AI answer conveys the exact same information as the original answer and accurately addresses the student's question.", score='good')

In [14]:
calc_price(usage)

{'input_cost': 0.000333,
 'output_cost': 0.00035099999999999997,
 'total_cost': 0.0006839999999999999}

In [32]:
def evaluate_aqa_croco(question,answer,llm_answer,model="gemini-3.1-flash-lite"):
    
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer,
        answer_llm=llm_answer
    )

    result,usage = llm_structured_retry(
        client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model
    )
    return result,usage

In [28]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gemini-3.1-flash-lite"):
    
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [25]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer conveys the exact same information as the original answer and accurately reflects the requirements for joining the course late and obtaining a certificate.', score='good')

In [33]:
eval_result, usage = evaluate_aqa_croco(
    question=rec["question"],
    answer=rec["answer_orig"],
    llm_answer=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer conveys the exact same information as the ground truth, confirming that late enrollment is allowed and noting the condition regarding the project submission deadline for certificate eligibility.', score='good')

In [37]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"],
        model = "gemini-3.5-flash"
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage
    

In [38]:
from concurrent.futures import ThreadPoolExecutor


In [39]:
try: 
    with ThreadPoolExecutor(max_workers=6) as pool:
        results = map_progress(pool, answers, judge_record)
except Exception as e : 
    print(e)
finally: 
    print("you are so poor fam")

  0%|          | 0/395 [00:00<?, ?it/s]

429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.5-flash\nPlease retry in 45.858165227s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-3.5-flash', 'location': 'global'}, 'quotaValu

In [84]:
def evaluate_aqa(question, answer_orig, answer_llm, model="tencent/hy3:free"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [85]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

In [87]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"],
        model = "prism-ml/Ternary-Bonsai-27B-gguf:together"
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [10]:
import os
from dotenv import load_dotenv as ld
from openai import OpenAI
ld()
client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.getenv("HF_TOKEN")
)

completion = client.chat.completions.create(
    model="prism-ml/Ternary-Bonsai-27B-gguf:together",
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France?"
        }
    ],
)



In [11]:
print(completion.choices[0].message.content)


The capital of France is Paris.
